In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
os.environ["KERAS_BACKEND"] = "torch"

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import bayesflow as bf

from numba import njit, prange

INFO:bayesflow:Using backend 'torch'
When using torch backend, we need to disable autograd by default to avoid excessive memory usage. Use

with torch.enable_grad():
    ...

in contexts where you need gradients (e.g. custom training loops).


## DDM simulator with regressed parameterization

In [4]:
@njit
def simulate_ddm_trial(
    v: float,
    a: float,
    tau: float,
    s_v: float,
    s_tau: float,
    decay: float = 0.0,
    z: float = 0.0,
    sigma: float = 1.0,
    dt: float = 0.001,
    max_steps: int = 10000,
    log_transform: bool = True,
):

    if log_transform:
        a = np.exp(a)
        tau = np.exp(tau)
        # s_v = np.exp(s_v)

    v_i = np.random.normal(v, s_v)

    tau_i = tau + np.random.uniform(-s_tau * tau, s_tau * tau)

    # initialize
    a0 = a if a > 1e-6 else 1e-6
    d  = decay if decay > 0.0 else 0.0

    x = z * a0
    t = tau_i

    for _ in range(max_steps):
        t += dt
        bound = a0 * np.exp(-d * t)
        if bound < 1e-3:
            bound = 1e-3
        x += v_i * dt + sigma * np.sqrt(dt) * np.random.normal()
        if x >= bound:
            return np.array([t, 1.0], dtype=np.float32)
        if x <= -bound:
            return np.array([t, 0.0], dtype=np.float32)
    # No decision within max_steps
    return np.array([-1.0, -1.0], dtype=np.float32)

In [5]:
@njit
def simulate_ddm_dataset_regressed(
    x: np.ndarray,                 # shape (num_obs,)
    beta_v0: float,
    beta_v1: float,
    beta_tau0: float,
    beta_tau1: float,
    a: float,
    s_v: float,
    s_tau: float,
    decay: float = 0.0,
    z: float = 0.0,
    sigma: float = 1.0,
    dt: float = 0.001,
    max_steps: int = 10000,
    log_transform: bool = True,
) -> np.ndarray:
    """
    Returns array of shape (num_obs, 2): [rt, choice] per trial.
    Uses *regression coefficients* as the inferred parameters (intercepts/slopes).
    """
    n = x.shape[0]
    out = np.empty((n, 2), dtype=np.float32)

    for i in range(n):
        v_i = beta_v0 + beta_v1 * x[i]
        tau_i = beta_tau0 + beta_tau1 * x[i]

        out[i, :] = simulate_ddm_trial(
            v=v_i,
            a=a,
            tau=tau_i,
            s_v=s_v,
            s_tau=s_tau,
            decay=decay,
            z=z,
            sigma=sigma,
            dt=dt,
            max_steps=max_steps,
            log_transform=log_transform,
        )

    return out

In [11]:
class RegressedDDMSimulator:
    """
    Batch simulator for a DDM where drift and non-decision time
    are regressed on a known trial-level covariate x.
    """

    def __init__(
        self,
        x_sampler,
        prior_sampler,
        log_transform: bool = True,
    ):
        """
        Parameters
        ----------
        x_sampler : callable
            Function x_sampler(num_obs) -> array (num_obs,)
        prior_sampler : callable
            Function prior_sampler() -> dict of global parameters
        log_transform : bool
            Passed to simulate_ddm_trial
        """
        self.x_sampler = x_sampler
        self.prior_sampler = prior_sampler
        self.log_transform = log_transform

    def sample(
        self,
        batch_size: int | tuple,
        num_obs: int = 500,
    ):
        """
        Sample a batch of simulated datasets.

        Parameters
        ----------
        batch_size : int or (int, int)
            Number of datasets, or range to sample from.
        num_obs : int
            Number of trials per dataset.

        Returns
        -------
        list[dict]
            Each element contains x, rt, choice, params.
        """
        # Resolve batch size
        if isinstance(batch_size, tuple):
            batch_size = batch_size[0]

        batch = []

        for _ in range(batch_size):
            # 1) Sample regressor
            x = self.x_sampler(num_obs).astype(np.float32)

            # 2) Sample global parameters
            params = self.prior_sampler()

            # 3) Simulate dataset
            sim = simulate_ddm_dataset_regressed(
                x=x,
                beta_v0=params["beta_v0"],
                beta_v1=params["beta_v1"],
                beta_tau0=params["beta_tau0"],
                beta_tau1=params["beta_tau1"],
                a=params["a"],
                s_v=params["s_v"],
                s_tau=params["s_tau"],
                decay=params.get("decay", 0.0),
                z=params.get("z", 0.0),
                sigma=params.get("sigma", 1.0),
                log_transform=self.log_transform,
            )

            batch.append({"rts": sim[:, 0], "choice": sim[:, 1]} | params)

        return batch

In [14]:
def x_sampler(num_obs):
    # Example: signed stimulus strength
    return np.random.uniform(-1.0, 1.0, size=num_obs)

In [15]:
def priors():
    return {
        "beta_v0": np.random.normal(0.0, 0.5),
        "beta_v1": np.random.normal(1.0, 0.5),
        "beta_tau0": np.random.normal(np.log(0.3), 0.2),
        "beta_tau1": np.random.normal(0.0, 0.2),
        "a": np.random.normal(np.log(1.5), 0.2),
        "s_v": np.random.uniform(0.0, 0.5),
        "s_tau": np.random.uniform(0.0, 0.3),
    }

In [23]:
num_obs = 500
simulator = RegressedDDMSimulator(x_sampler=x_sampler, prior_sampler=priors)

In [24]:
samples = simulator.sample(batch_size=2)

In [25]:
samples

[{'x': array([ 0.23836632,  0.46460116, -0.04293522, -0.1350616 ,  0.10527643,
         -0.23241743, -0.18167996,  0.24881844, -0.53810054, -0.96168524,
          0.07416908,  0.8018485 ,  0.47305804, -0.255445  , -0.26159513,
         -0.55792284, -0.50409937,  0.22076657,  0.42065194, -0.8440888 ,
          0.29936948,  0.6532344 ,  0.3241233 ,  0.8223184 , -0.05396214,
          0.2560136 ,  0.2811526 , -0.2934609 ,  0.9846387 ,  0.25224108,
          0.96133804,  0.7693257 , -0.72136366, -0.5911397 ,  0.14089523,
         -0.24691601,  0.36698148,  0.34760538, -0.16901004, -0.97600055,
         -0.7473585 ,  0.10148567,  0.68559104, -0.32182378, -0.2805379 ,
          0.8256989 , -0.1636975 ,  0.70012975, -0.21470882,  0.10982266,
          0.07329002, -0.67786115,  0.35060552,  0.7319444 ,  0.14771055,
         -0.22993582, -0.5266809 , -0.656095  , -0.0749813 ,  0.5974777 ,
          0.04424001,  0.38174444,  0.78432024,  0.38009587,  0.36916244,
          0.20446129, -0.83540916